In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk import pos_tag

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

# --- Load data ---
DATA_PATH = "/content/drive/MyDrive/Dataset (Decode Labs)/IMDB Dataset.csv"
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print(df['sentiment'].value_counts())
df.head()

# --- Text pre-processing pipeline ---
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
# Critical: exclude negations from stop words, so "not good" isn't destroyed
negations = {'not', 'no', 'nor', "n't", 'never', 'none'}
stop_words = stop_words - negations

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return 'a'  # adjective
    elif tag.startswith('V'):
        return 'v'  # verb
    elif tag.startswith('R'):
        return 'r'  # adverb
    else:
        return 'n'  # default: noun

def preprocess_text(text):
    # Remove HTML tags (this dataset has <br> tags)
    text = re.sub(r'<.*?>', ' ', text)
    # Lowercase
    text = text.lower()
    # Remove non-alphabetic characters
    text = re.sub(r'[^a-z\s]', ' ', text)
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords (except negations)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    # POS-guided lemmatization
    pos_tags = pos_tag(tokens)
    lemmatized = [lemmatizer.lemmatize(t, get_wordnet_pos(pos)) for t, pos in pos_tags]
    return ' '.join(lemmatized)

print("Preprocessing reviews... this may take a few minutes on the full 50k dataset")
df['cleaned_review'] = df['review'].apply(preprocess_text)
print("Done.")
df[['review', 'cleaned_review']].head()

# --- Encode target ---
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# --- Train/test split ---
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['cleaned_review'], df['label'], test_size=0.2, stratify=df['label'], random_state=42
)

# --- TF-IDF vectorization ---
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=10000,
    min_df=2,
    ngram_range=(1, 2)  # unigrams + bigrams, to capture things like "not good"
)

X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf = vectorizer.transform(X_test_text)

print("TF-IDF matrix shape (train):", X_train_tfidf.shape)
print("TF-IDF matrix shape (test):", X_test_tfidf.shape)

# --- Train Naive Bayes classifier ---
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

nb_model = MultinomialNB(alpha=1.0)  # alpha=1.0 is Laplace smoothing
nb_model.fit(X_train_tfidf, y_train)

y_pred_nb = nb_model.predict(X_test_tfidf)

print("--- Naive Bayes Results ---")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("Precision:", precision_score(y_test, y_pred_nb))
print("Recall:", recall_score(y_test, y_pred_nb))
print("F1 Score:", f1_score(y_test, y_pred_nb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

# --- Train SVM classifier (for comparison) ---
from sklearn.svm import LinearSVC

svm_model = LinearSVC(random_state=42)
svm_model.fit(X_train_tfidf, y_train)

y_pred_svm = svm_model.predict(X_test_tfidf)

print("--- SVM Results ---")
print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print("Precision:", precision_score(y_test, y_pred_svm))
print("Recall:", recall_score(y_test, y_pred_svm))
print("F1 Score:", f1_score(y_test, y_pred_svm))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

# --- Test on a few custom examples ---
sample_reviews = [
    "This movie was absolutely fantastic, I loved every minute of it!",
    "Terrible film, complete waste of time. Not good at all.",
    "It was okay, not the best but not the worst either."
]

sample_cleaned = [preprocess_text(r) for r in sample_reviews]
sample_tfidf = vectorizer.transform(sample_cleaned)
sample_preds = nb_model.predict(sample_tfidf)

for review, pred in zip(sample_reviews, sample_preds):
    sentiment = "Positive" if pred == 1 else "Negative"
    print(f"'{review}' -> {sentiment}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


Shape: (50000, 2)
sentiment
positive    25000
negative    25000
Name: count, dtype: int64
Preprocessing reviews... this may take a few minutes on the full 50k dataset
Done.
TF-IDF matrix shape (train): (40000, 10000)
TF-IDF matrix shape (test): (10000, 10000)
--- Naive Bayes Results ---
Accuracy: 0.8693
Precision: 0.8561234329797492
Recall: 0.8878
F1 Score: 0.871674030436917
Confusion Matrix:
 [[4254  746]
 [ 561 4439]]
              precision    recall  f1-score   support

           0       0.88      0.85      0.87      5000
           1       0.86      0.89      0.87      5000

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000

--- SVM Results ---
Accuracy: 0.8891
Precision: 0.8880909634949132
Recall: 0.8904
F1 Score: 0.889243982822331
Confusion Matrix:
 [[4439  561]
 [ 548 4452]]
              precision    recall  f1-score   support

           0       0.89      0.89     